# Clinical RAG System: Sickle Cell Disease (Three-Document Corpus, Azure-Backed)

A citation-aware Retrieval-Augmented Generation system built from three sources:

1. **NHLBI (2014)**: Evidence-Based Management of Sickle Cell Disease: Expert Panel Report, 2014.
2. **JCM (2024)**: Clinical Insights into Sickle Cell Disease, a multicenter retrospective analysis across age groups.
3. **WHO (2026)**: WHO consolidated guidelines for the management of common childhood illness: management of sickle-cell disease in children and adolescents.

Vector storage and hybrid retrieval run on **Azure AI Search** rather than a local, session-scoped
Chroma instance. The corpus is indexed once; subsequent sessions query the existing Azure index
directly, without re-embedding or re-uploading the documents.

**Safety:** Educational use only. The system does not diagnose, prescribe, or replace a clinician.


## 1. Setup

In [1]:
!pip install -q langchain langchain-community langchain-text-splitters langchain-chroma chromadb fastembed groq pypdf jsonschema azure-search-documents azure-core

# langchain-chroma and chromadb are retained only for the local, one-time chunk-size
# experiment in Section 15. The production retrieval path uses Azure AI Search and does
# not depend on Chroma.

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.4/83.4 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 71.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 382.9/382.9 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 352.1/352.1 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.9/220.9 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 97.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 2. Upload the PDF

Download the report first from the link above, then upload it here.

In [2]:
from google.colab import files

print("Upload 1 of 3 -- NHLBI 2014 SCD Guideline PDF (56-364NFULL.pdf)")
uploaded1 = files.upload()
nhlbi_path = next(iter(uploaded1))
print(f"  loaded: {nhlbi_path}")

print("\nUpload 2 of 3 -- JCM 2024 research paper PDF")
uploaded2 = files.upload()
jcm_path = next(iter(uploaded2))
print(f"  loaded: {jcm_path}")

print("\nUpload 3 of 3 -- WHO 2026 childhood SCD guideline PDF (9789240122666-eng_RAG_clean_glossary_start.pdf)")
uploaded3 = files.upload()
who_path = next(iter(uploaded3))
print(f"  loaded: {who_path}")


Upload 1 of 3 -- NHLBI 2014 SCD Guideline PDF (56-364NFULL.pdf)


Saving 56-364NFULL.pdf to 56-364NFULL.pdf
  loaded: 56-364NFULL.pdf

Upload 2 of 3 -- JCM 2024 research paper PDF


Saving jcm-13-07224_RAG_clean_v3_abstract_only_page1.pdf to jcm-13-07224_RAG_clean_v3_abstract_only_page1.pdf
  loaded: jcm-13-07224_RAG_clean_v3_abstract_only_page1.pdf

Upload 3 of 3 -- WHO 2026 childhood SCD guideline PDF (9789240122666-eng_RAG_clean_glossary_start.pdf)


Saving 9789240122666-eng_RAG_clean_glossary_start.pdf to 9789240122666-eng_RAG_clean_glossary_start.pdf
  loaded: 9789240122666-eng_RAG_clean_glossary_start.pdf


## 3. Load pages and attach citation metadata

In [11]:
from langchain_community.document_loaders import PyPDFLoader

DOC_CONFIGS = [
    {
        'path':        nhlbi_path,
        'document_id': 'nhlbi-scd-2014',
        'title':       'Evidence-Based Management of Sickle Cell Disease: Expert Panel Report, 2014',
        'citation':    'National Heart, Lung, and Blood Institute (2014). Evidence-Based Management of Sickle Cell Disease: Expert Panel Report, 2014.',
    },
    {
        'path':        jcm_path,
        'document_id': '10_3390_jcm13237224',
        'title':       'Clinical Insights into Sickle Cell Disease: A Comprehensive Multicenter Retrospective Analysis of Clinical Characteristics and Outcomes Across Different Age Groups',
        'citation':    'Almarghalani DA, Alotaibi RA, Alzlami TT, Alhumaidi OF, Alharthi NM, Alboqami FM, Almehmadi KA, Miski SF, Alshahrani A, Alamri FF, Alsolami K, Doman SM, Alhamdi MT, Zubaid A, Aloufi WS. Clinical Insights into Sickle Cell Disease: A Comprehensive Multicenter Retrospective Analysis of Clinical Characteristics and Outcomes Across Different Age Groups. J Clin Med. 2024;13(23):7224. doi:10.3390/jcm13237224.',
    },
    {
        'path':        who_path,
        'document_id': '9789240122666',
        'title':       'WHO consolidated guidelines for the management of common childhood illness: management of sickle-cell disease in children and adolescents',
        'citation':    'World Health Organization. WHO consolidated guidelines for the management of common childhood illness: management of sickle-cell disease in children and adolescents. Geneva: World Health Organization; 2026. ISBN 978-92-4-012266-6.',
    },
]

pages = []
for cfg in DOC_CONFIGS:
    loader = PyPDFLoader(cfg['path'])
    doc_pages = loader.load()
    for page in doc_pages:
        page.metadata.update({
            'document_id': cfg['document_id'],
            'title':       cfg['title'],
            'citation':    cfg['citation'],
            'page_number': page.metadata.get('page', 0) + 1,
        })
    pages.extend(doc_pages)
    print(f"Loaded {len(doc_pages):3d} pages from [{cfg['document_id']}]")

print(f"\nTotal pages across all three documents: {len(pages)}")
print(pages[0].metadata)


Loaded 105 pages from [nhlbi-scd-2014]
Loaded   9 pages from [10_3390_jcm13237224]
Loaded  95 pages from [9789240122666]

Total pages across all three documents: 209
{'producer': 'pypdf', 'creator': 'PyPDF', 'creationdate': '', 'source': '56-364NFULL.pdf', 'total_pages': 105, 'page': 0, 'page_label': '1', 'document_id': 'nhlbi-scd-2014', 'title': 'Evidence-Based Management of Sickle Cell Disease: Expert Panel Report, 2014', 'citation': 'National Heart, Lung, and Blood Institute (2014). Evidence-Based Management of Sickle Cell Disease: Expert Panel Report, 2014.', 'page_number': 1}


## 4. Chunk

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=['\n\n', '\n', '. ', ' ', '']
)
chunks = splitter.split_documents(pages)

# Assign stable chunk IDs scoped to each document
doc_counters = {}
for chunk in chunks:
    doc_id = chunk.metadata['document_id']
    doc_counters[doc_id] = doc_counters.get(doc_id, 0) + 1
    chunk.metadata['chunk_id'] = f"{doc_id}-CH-{doc_counters[doc_id]:04d}"

print(f'Created {len(chunks)} chunks from {len(pages)} pages')
for doc_id, count in doc_counters.items():
    print(f'  {doc_id}: {count} chunks')
print()
print(chunks[0].metadata)


Created 1019 chunks from 209 pages
  nhlbi-scd-2014: 510 chunks
  10_3390_jcm13237224: 61 chunks
  9789240122666: 448 chunks

{'producer': 'pypdf', 'creator': 'PyPDF', 'creationdate': '', 'source': '56-364NFULL.pdf', 'total_pages': 105, 'page': 0, 'page_label': '1', 'document_id': 'nhlbi-scd-2014', 'title': 'Evidence-Based Management of Sickle Cell Disease: Expert Panel Report, 2014', 'citation': 'National Heart, Lung, and Blood Institute (2014). Evidence-Based Management of Sickle Cell Disease: Expert Panel Report, 2014.', 'page_number': 1, 'chunk_id': 'nhlbi-scd-2014-CH-0001'}


Spot-check a few chunks before moving on:

In [13]:
for c in chunks[:3]:
    print(c.metadata['chunk_id'], '| page', c.metadata['page_number'])
    print(c.page_content[:800].replace('\n', ' '))
    print()

nhlbi-scd-2014-CH-0001 | page 1
Chapter 1:  Introduction and  Methodology  These guidelines were developed by an expert panel composed of health care professionals with expertise in  family medicine, general internal medicine, adult and pediatric hematology, psychiatry, transfusion medicine,  obstetrics and gynecology, emergency department nursing, and evidence-based medicine.  Panel members were  selected by the National Heart, Lung, and Blood Institute’s (NHLBI’s) leadership.  The purpose of these guidelines is to help people living with sickle cell disease (SCD) receive appropriate care  by providing the best science-based recommendations to guide practice decisions.  The target audience is  primary care providers and other clinicians, nurses, and staff who provide emergency or continuity care to

nhlbi-scd-2014-CH-0002 | page 1
primary care providers and other clinicians, nurses, and staff who provide emergency or continuity care to  individuals with SCD.   NHLBI sponsored the deve

In [ ]:
!pip install langchain_huggingface



```
# This is formatted as code
```

## 5. Azure AI Search: connection and index schema

Index creation is idempotent (`create_or_update_index`), so this cell is safe to re-run without
duplicating or corrupting an existing index. The Azure AI Search resource itself is created once,
in the Azure Portal, outside this notebook.

In [27]:
from getpass import getpass
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex, SimpleField, SearchableField, SearchField,
    SearchFieldDataType, VectorSearch, HnswAlgorithmConfiguration,
    VectorSearchProfile,
)
from azure.search.documents import SearchClient

AZURE_SEARCH_ENDPOINT = input('Azure AI Search endpoint (https://<service-name>.search.windows.net): ').strip()
AZURE_SEARCH_KEY = getpass('Azure AI Search admin key: ')
INDEX_NAME = 'creativa-hackathon-vb'

EMBEDDING_DIM = 768  # NeuML/biomedbert-base-embeddings output dimension

index_client = SearchIndexClient(AZURE_SEARCH_ENDPOINT, AzureKeyCredential(AZURE_SEARCH_KEY))

fields = [
    SimpleField(name='chunk_id', type=SearchFieldDataType.String, key=True),
    SimpleField(name='document_id', type=SearchFieldDataType.String, filterable=True),
    SimpleField(name='title', type=SearchFieldDataType.String),
    SimpleField(name='citation', type=SearchFieldDataType.String),
    SimpleField(name='section', type=SearchFieldDataType.String, filterable=True),
    SimpleField(name='page_number', type=SearchFieldDataType.Int32, filterable=True),
    SearchableField(name='content', type=SearchFieldDataType.String),
    SearchField(
        name='content_vector',
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True,
        vector_search_dimensions=EMBEDDING_DIM,
        vector_search_profile_name='scd-vector-profile',
    ),
]

vector_search = VectorSearch(
    algorithms=[HnswAlgorithmConfiguration(name='scd-hnsw')],
    profiles=[VectorSearchProfile(name='scd-vector-profile', algorithm_configuration_name='scd-hnsw')],
)

index = SearchIndex(name=INDEX_NAME, fields=fields, vector_search=vector_search)
index_client.create_or_update_index(index)

search_client = SearchClient(AZURE_SEARCH_ENDPOINT, INDEX_NAME, AzureKeyCredential(AZURE_SEARCH_KEY))
print(f"Index '{INDEX_NAME}' ready.")


Azure AI Search endpoint (https://<service-name>.search.windows.net): https://creativa-hackathon-vb.search.windows.net
Azure AI Search admin key: ··········
Index 'creativa-hackathon-vb' ready.


## 6. Embeddings and Azure AI Search indexing

Chunks are embedded locally with the same model used throughout the project, then uploaded to
the Azure index in batches. This step needs to run only once per corpus version; re-running it
re-embeds and re-uploads, which is safe but unnecessary if the index already reflects the current
chunk set.

`retrieve_with_similarity` is defined here as the production retrieval function used by every
downstream section, Day 2 and Day 3 alike. It issues a single Azure AI Search query combining
keyword (BM25) and vector search, merged server-side via Reciprocal Rank Fusion, which is Azure's
native hybrid search. This replaces the manually implemented BM25 index from the earlier Section
20 approach, and resolves the gap recorded in the Day 2 final configuration, where a hybrid
strategy was specified but not actually wired into retrieval.

Retrieved results are wrapped in a small `AzureChunk` object exposing `.page_content` and
`.metadata`, matching the interface every existing downstream cell (evidence panel, Top-K
comparison, evaluation loop, citation verification, generation) already expects. No other cell
in this notebook needs to change as a result of this backend swap.

In [28]:
from langchain_huggingface import HuggingFaceEmbeddings
from azure.search.documents.models import VectorizedQuery

# Biomedical embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name='NeuML/biomedbert-base-embeddings',
    model_kwargs={
        'device': 'cpu'
    },
    encode_kwargs={
        'normalize_embeddings': True
    }
)

raw_texts = [c.page_content for c in chunks]

print(f"Embedding {len(raw_texts)} chunks...")

vectors = embedding_model.embed_documents(raw_texts)

# Sanity check
print(f"Embedding dimension: {len(vectors[0])}")
assert len(vectors[0]) == 768, (
    f"Expected 768 dimensions, got {len(vectors[0])}"
)

documents_to_upload = []

for chunk, vector in zip(chunks, vectors):
    m = chunk.metadata

    documents_to_upload.append({
        'chunk_id': m['chunk_id'],
        'document_id': m['document_id'],
        'title': m['title'],
        'citation': m['citation'],
        'section': (
            guess_section(chunk.page_content)
            if 'guess_section' in globals()
            else 'Unknown'
        ),
        'page_number': m['page_number'],
        'content': chunk.page_content,
        'content_vector': vector,
    })

batch_size = 500

for i in range(0, len(documents_to_upload), batch_size):
    batch = documents_to_upload[i:i + batch_size]

    result = search_client.upload_documents(
        documents=batch
    )

    succeeded = sum(r.succeeded for r in result)

    print(
        f"Uploaded batch {i // batch_size + 1}: "
        f"{succeeded}/{len(batch)} succeeded"
    )


class AzureChunk:
    """Thin wrapper so Azure results expose the same
    .page_content / .metadata interface used downstream."""

    def __init__(self, result):
        self.page_content = result['content']

        self.metadata = {
            'chunk_id': result['chunk_id'],
            'document_id': result['document_id'],
            'title': result.get('title', ''),
            'citation': result['citation'],
            'section': result.get('section', 'Unknown'),
            'page_number': result['page_number'],
        }


def retrieve_with_similarity(
    question: str,
    k: int = 5
):
    query_vector = embedding_model.embed_query(question)

    # Sanity check for query embedding
    assert len(query_vector) == 768, (
        f"Expected 768 dimensions, got {len(query_vector)}"
    )

    vector_query = VectorizedQuery(
        vector=query_vector,
        k_nearest_neighbors=k,
        fields='content_vector',
    )

    results = search_client.search(
        search_text=question,           # BM25 lexical retrieval
        vector_queries=[vector_query],  # semantic/vector retrieval
        select=[
            'chunk_id',
            'document_id',
            'title',
            'citation',
            'section',
            'page_number',
            'content'
        ],
        top=k,
    )

    return [
        (AzureChunk(r), r['@search.score'])
        for r in results
    ]


print(
    'Azure AI Search indexing complete '
    'with PubMedBERT embeddings.'
)
print('Hybrid retrieval is enabled.')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding 1019 chunks...
Embedding dimension: 768
Uploaded batch 1: 500/500 succeeded
Uploaded batch 2: 500/500 succeeded
Uploaded batch 3: 19/19 succeeded
Azure AI Search indexing complete with PubMedBERT embeddings.
Hybrid retrieval is enabled.


## 7. Retrieval test, no LLM call needed

Confirms the Azure-backed hybrid retrieval returns relevant passages before any LLM budget is spent.

In [29]:
test_question = 'When should hydroxyurea therapy be started in adults with sickle cell anemia?'
retrieved = retrieve_with_similarity(test_question)
for rank, (d, score) in enumerate(retrieved, start=1):
    print(f"Rank {rank} | {d.metadata['chunk_id']} | page {d.metadata['page_number']} | similarity: {score:.4f}")
    print(d.page_content[:300], '\n')

Rank 1 | nhlbi-scd-2014-CH-0405 | page 78 | similarity: 0.0328
 A lack of increase in MCV and/or HbF is not an indication to discontinue therapy. 
 For the patient who has a clinical response, long-term hydroxyurea therapy is indicated. 
 Hydroxyurea therapy should be continued during hospitalizations or illness. 
EVIDENCE-BASED MANAGEMENT OF SICKLE CELL DIS 

Rank 2 | nhlbi-scd-2014-CH-0384 | page 75 | similarity: 0.0323
Exhibit 12. Evidence Profile—Evidence of Efficacy/Effectiveness for Children and Adults With 
Sickle Cell Anemia (Hydroxyurea Versus Usual Care) 
Outcome Quality of the Evidence Treatment Effect 
Pain crises High Statistically significant benefit 
Acute chest syndrome Moderate Statistically signific 

Rank 3 | nhlbi-scd-2014-CH-0374 | page 73 | similarity: 0.0313
for hydroxyurea in patients with SCA. 
Evidence of Efficacy/Effectiveness 
Summary of Evidence in Adults With SCA 
The Multicenter Study of Hydroxyurea in Patients With Sickle Cell Anemia (MSH) was a rando

## 8. Secure LLM connection (Groq)

In [30]:
import os
from getpass import getpass
from groq import Groq

if 'GROQ_API_KEY' not in os.environ:
    os.environ['GROQ_API_KEY'] = getpass('Enter your Groq API key: ')

client = Groq(api_key=os.environ['GROQ_API_KEY'])
LLM_MODEL = 'openai/gpt-oss-120b'

## 9. Clinical prompt and citation formatter

In [31]:
SYSTEM_PROMPT = '''You are a clinical education assistant specializing in sickle cell disease.
Use ONLY the supplied context to answer. If the context is insufficient, say exactly:
"The provided sources do not contain enough information to answer that."
Do not diagnose, prescribe, recommend drug doses, or select personalized treatment for any individual.
For concerning symptoms, advise assessment by a qualified clinician.
Every factual statement must end with a citation in this format: [Source: <citation>, p. <page_number>].'''

def format_docs(scored_docs):
    parts = []
    for doc, score in scored_docs:
        parts.append(f"({doc.metadata['citation']}, p. {doc.metadata['page_number']})\n{doc.page_content}")
    return '\n\n---\n\n'.join(parts)

## 10. Ask questions with retrieved evidence

In [32]:
def ask_clinical_rag(question: str, k: int = 7):
    scored_docs = retrieve_with_similarity(question, k=k)
    context = format_docs(scored_docs)

    response = client.chat.completions.create(
        model=LLM_MODEL,
        temperature=0.1,
        max_tokens=700,
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': f'Context:\n\n{context}\n\nQuestion: {question}'},
        ],
    )
    answer_text = response.choices[0].message.content

    return {
        'answer': answer_text,
        'retrieved_sources': [
            {
                'chunk_id': doc.metadata['chunk_id'],
                'page': doc.metadata['page_number'],
                'similarity': round(score, 4),
            }
            for doc, score in scored_docs
        ],
    }

In [33]:
result = ask_clinical_rag('When should hydroxyurea therapy be started in adults with sickle cell anemia?')
print(result['answer'])
print('\nRetrieved sources:')
for source in result['retrieved_sources']:
    print(source)

Hydroxyurea should be initiated in adults with sickle cell anemia who have experienced **three or more moderate‑to‑severe sickle‑cell‑related pain crises within a 12‑month period**【Source: National Heart, Lung, and Blood Institute (2014). Evidence-Based Management of Sickle Cell Disease: Expert Panel Report, 2014., p. 77】.

Retrieved sources:
{'chunk_id': 'nhlbi-scd-2014-CH-0405', 'page': 78, 'similarity': 0.0328}
{'chunk_id': 'nhlbi-scd-2014-CH-0384', 'page': 75, 'similarity': 0.0323}
{'chunk_id': 'nhlbi-scd-2014-CH-0374', 'page': 73, 'similarity': 0.0313}
{'chunk_id': 'nhlbi-scd-2014-CH-0396', 'page': 77, 'similarity': 0.0306}
{'chunk_id': 'nhlbi-scd-2014-CH-0016', 'page': 3, 'similarity': 0.028}
{'chunk_id': 'nhlbi-scd-2014-CH-0373', 'page': 73, 'similarity': 0.0278}
{'chunk_id': '9789240122666-CH-0160', 'page': 35, 'similarity': 0.0194}


# Day 2 — Retrieval Optimization

Day 1 built the RAG pipeline for the NHLBI Sickle Cell Disease report. Day 2 keeps that work
and improves only the **retrieval layer**.

> **Rule for today:** do not optimize the prompt before verifying the evidence.

We will:
1. Confirm the Day 1 handoff (retrieval + metadata still work).
2. Add a lightweight section tag for display purposes.
3. Tune `Top-K` (3 / 5 / 10).
4. Compare chunk size / overlap configurations.
5. Build a 15–20 question evaluation set (5 categories).
6. Manually label retrieved chunks as relevant / not relevant.
7. Calculate Precision@3 and Precision@5.
8. Log real failure cases using the failure-mode table.
9. (Optional) Try keyword / hybrid search on one question.
10. Choose and justify a final retrieval configuration.

> Run all Day 1 cells first. Day 2 reuses the existing `pages`, `chunks`, `embedding_model`,
> `search_client`, and `retrieve_with_similarity` from Day 1, all backed by Azure AI Search — it does not rebuild them.

## 11. Day 1 handoff check

Confirm retrieval returns chunk text plus citation metadata **before** tuning anything. This checks evidence selection, not answer writing — no LLM call here.

In [34]:
handoff_question = 'When should hydroxyurea therapy be started in adults with sickle cell anemia?'

for rank, (doc, score) in enumerate(retrieve_with_similarity(handoff_question, k=5), start=1):
    print(f"Rank {rank} | Score {score:.4f} | Page {doc.metadata['page_number']} | {doc.metadata['chunk_id']}")
    print(doc.page_content[:350].replace('\n', ' '))
    print('-' * 100)

Rank 1 | Score 0.0328 | Page 78 | nhlbi-scd-2014-CH-0405
 A lack of increase in MCV and/or HbF is not an indication to discontinue therapy.   For the patient who has a clinical response, long-term hydroxyurea therapy is indicated.   Hydroxyurea therapy should be continued during hospitalizations or illness.  EVIDENCE-BASED MANAGEMENT OF SICKLE CELL DISEASE: EXPERT PANEL REPORT, 2014 78
----------------------------------------------------------------------------------------------------
Rank 2 | Score 0.0323 | Page 75 | nhlbi-scd-2014-CH-0384
Exhibit 12. Evidence Profile—Evidence of Efficacy/Effectiveness for Children and Adults With  Sickle Cell Anemia (Hydroxyurea Versus Usual Care)  Outcome Quality of the Evidence Treatment Effect  Pain crises High Statistically significant benefit  Acute chest syndrome Moderate Statistically significant benefit  Hemoglobin level, fetal hemoglobin le
---------------------------------------------------------------------------------------------------

## 12. Section tagging (for display only)

A lightweight keyword-based tagger, extended to cover topics introduced by all three source
documents: the NHLBI 2014 guideline, the JCM 2024 paper, and the WHO 2026 pediatric guideline.
Order matters: the first matching pattern wins, so more specific patterns are listed before
broader ones they could otherwise be absorbed into.

In [21]:
import re

SECTION_KEYWORDS = [
    ('Hydroxyurea Therapy',        [r'hydroxyurea']),
    ('Stroke Prevention / TCD',    [r'transcranial doppler', r'\btcd\b', r'stroke']),
    ('Acute Chest Syndrome',       [r'acute chest syndrome']),
    ('Pain Management',            [r'vaso-?occlusive', r'pain crisis', r'analgesi']),
    ('Transfusion Therapy',        [r'transfusion', r'alloimmuni']),
    ('Priapism',                   [r'priapism']),
    ('Pregnancy / Reproductive',   [r'pregnan', r'contracept']),
    ('Renal Complications',        [r'nephropathy', r'proteinuria', r'\begfr\b', r'chronic kidney', r'creatinine']),
    ('Ophthalmologic Screening',   [r'retinopathy', r'ophthalmolog']),
    ('Pulmonary Hypertension',     [r'pulmonary hypertension']),
    ('Mental Health / Screening',  [r'depression', r'anxiety', r'neurocognitive']),
    # JCM 2024 paper topics
    ('Age-Group Epidemiology',     [r'age group', r'adolescent', r'middle.aged']),
    ('Splenic Disease',            [r'splenic', r'splenomegaly', r'splenic sequestration']),
    ('Hospital Utilisation',       [r'hospital stay', r'icu admission', r'length of stay', r'readmission']),
    ('Treatment Adherence',        [r'adherence', r'compliance', r'opioid use']),
    # WHO 2026 pediatric guideline topics
    ('Newborn Screening',          [r'newborn screening', r'neonatal screen']),
    ('Malaria Prophylaxis',        [r'malaria prophylaxis', r'antimalarial', r'malaria co-?endem']),
    ('Dactylitis / Hand-Foot',     [r'dactylitis', r'hand-foot syndrome']),
    ('Nutrition / Growth',         [r'growth monitoring', r'malnutrition', r'growth faltering', r'nutritional status']),
    ('Iron Supplementation',       [r'iron supplement', r'iron deficiency']),
    ('Danger Signs / Referral',    [r'danger sign', r'refer(ral)? to (hospital|higher level)', r'urgent referral']),
    ('Primary / Community Care',   [r'community health worker', r'primary (health )?care level', r'\bimci\b']),
    ('GRADE Evidence Strength',    [r'\bgrade\b.{0,20}(evidence|recommendation)', r'strong recommendation', r'conditional recommendation']),
    ('Pediatric Dosing',           [r'mg/kg', r'pediatric dos', r'dose.{0,20}(child|infant)']),
    ('Infection Prevention',       [r'pneumococcal', r'penicillin prophylaxis', r'immuniz', r'vaccin']),
    ('Folic Acid Supplementation', [r'folic acid']),
]

def guess_section(text: str) -> str:
    lowered = text.lower()
    for label, patterns in SECTION_KEYWORDS:
        if any(re.search(p, lowered) for p in patterns):
            return label
    return 'General / Unclassified'

print('Section tagger ready.')


Section tagger ready.


## 13. Evidence panel

Shows exactly what the LLM would see, with full traceability: score, document, page, guessed section, and chunk ID.

In [35]:
def show_evidence_panel(question: str, k: int = 5):
    print('CLINICAL QUERY:', question)
    print('=' * 110)
    for rank, (doc, score) in enumerate(retrieve_with_similarity(question, k=k), start=1):
        meta = doc.metadata
        print(
            f"Chunk {rank} | Score {score:.4f} | Doc {meta['document_id']} | "
            f"Page {meta['page_number']} | Section: {guess_section(doc.page_content)} | {meta['chunk_id']}"
        )
        print(doc.page_content[:450].replace('\n', ' '))
        print('-' * 110)

show_evidence_panel('When should hydroxyurea therapy be started in adults with sickle cell anemia?', k=5)

CLINICAL QUERY: When should hydroxyurea therapy be started in adults with sickle cell anemia?
Chunk 1 | Score 0.0328 | Doc nhlbi-scd-2014 | Page 78 | Section: Hydroxyurea Therapy | nhlbi-scd-2014-CH-0405
 A lack of increase in MCV and/or HbF is not an indication to discontinue therapy.   For the patient who has a clinical response, long-term hydroxyurea therapy is indicated.   Hydroxyurea therapy should be continued during hospitalizations or illness.  EVIDENCE-BASED MANAGEMENT OF SICKLE CELL DISEASE: EXPERT PANEL REPORT, 2014 78
--------------------------------------------------------------------------------------------------------------
Chunk 2 | Score 0.0323 | Doc nhlbi-scd-2014 | Page 75 | Section: Hydroxyurea Therapy | nhlbi-scd-2014-CH-0384
Exhibit 12. Evidence Profile—Evidence of Efficacy/Effectiveness for Children and Adults With  Sickle Cell Anemia (Hydroxyurea Versus Usual Care)  Outcome Quality of the Evidence Treatment Effect  Pain crises High Statistically significant b

## 14. Tune Top-K

- Small `k`: focused, may miss useful evidence.
- Large `k`: better coverage, but risks noise and duplicate chunks.

Compare `k = 3, 5, 10` on at least three questions. Read the chunks — don't decide from the score alone.

In [ ]:
def compare_top_k(question, k_values=(3, 5, 10)):
    for k in k_values:
        print(f'\n========== TOP-K = {k}  |  {question} ==========')
        for rank, (doc, score) in enumerate(retrieve_with_similarity(question, k=k), start=1):
            print(
                f"{rank}. score={score:.4f} | page={doc.metadata['page_number']} "
                f"| section={guess_section(doc.page_content)} | chunk={doc.metadata['chunk_id']}"
            )
            print(doc.page_content[:220].replace('\n', ' '))

topk_test_questions = [
    'When should hydroxyurea therapy be started in adults with sickle cell anemia?',
    'How is stroke risk screened in children with sickle cell disease?',
    'What vaccinations are recommended for infection prevention in sickle cell disease?',
]

for q in topk_test_questions:
    compare_top_k(q)


========== TOP-K = 3  |  When should hydroxyurea therapy be started in adults with sickle cell anemia? ==========
1. score=0.8635 | page=77 | section=Hydroxyurea Therapy | chunk=nhlbi-scd-2014-CH-0397
2. In adults with SCA who have three or more sickle cell-associated moderate to severe pain crises in a 12-month  period, treat with hydroxyurea.  (Strong Recommendation, High-Quality Evidence)  3. In adults with SCA who
2. score=0.8424 | page=71 | section=Hydroxyurea Therapy | chunk=nhlbi-scd-2014-CH-0361
Chapter 5: Hydroxyurea Therapy in the Management of Sickle Cell Disease  Introduction  This chapter addresses the use of hydroxyurea (also called hydroxycarbamide) in adults and children who have  SCD.  Hydroxyurea can r
3. score=0.8340 | page=73 | section=Hydroxyurea Therapy | chunk=nhlbi-scd-2014-CH-0374
for hydroxyurea in patients with SCA.  Evidence of Efficacy/Effectiveness  Summary of Evidence in Adults With SCA  The Multicenter Study of Hydroxyurea in Patients With Sickle Cell A

### Top-K checkpoint

Answer directly (as text, in this cell or a new one):

1. Does Top-3 contain enough evidence for these questions?
2. Does Top-10 add useful evidence, or mostly noise and repetition?
3. Which `k` would you choose for this report, and why?

1. Does Top-3 contain enough evidence?

It depends heavily on the question. For hydroxyurea (rank1=0.8635, rank2=0.8425), Top-3 is nearly sufficient — ranks 1–2 directly state the recommendation and its clinical context. But rank 3 is already a bibliography citation, not real content. Stroke screening is similar: rank 1 is a strong direct hit, but ranks 2–3 are reference-list entries — so effectively only 1 of 3 chunks carries evidence. Vaccination is the outlier and the real problem case: rank 1 is a partial/fragmentary sentence, and ranks 2–3 are bare PDF page-footer text ("112 EVIDENCE-BASED MANAGEMENT OF SICKLE CELL DISEASE..."), with zero actual clinical content. Top-3 fails outright for this question.

2. Does Top-10 add useful evidence, or mostly noise?

Mostly noise for two of three questions — the tail of Top-10 for hydroxyurea and stroke is dominated by repeated bibliography/citation chunks (author lists, journal names), which is repetition, not new evidence. But for the vaccination question, Top-10 is doing real work: the actual pneumococcal-vaccination recommendation text doesn't show up until rank 6 ("Assure that people of all ages with SCD have been vaccinated against Streptococcus pneumoniae...") and rank 7 (infant vaccination schedule). Without going past Top-5, the system would return zero usable evidence for that question.

3. Which k would you choose, and why?

k=5 as a general default — it captures the strong direct evidence for hydroxyurea and stroke without much added noise. But the vaccination case shows k alone isn't the real fix: the correct answer sits at rank 6–7 not because k was too small, but because low-value chunks (bibliography entries, bare page-footer text) are crowding out relevant content in the top ranks. Raising k to 10 papers over that problem with more tokens rather than solving it. The better fix — worth logging as a real failure case under "Correct chunk ranked too low" — is extending the TOC/reference-page filtering you already built in Day 1 to also strip bibliography and running-header/footer chunks before they ever enter the index, rather than just increasing k.

## 15. Compare chunk size and overlap (local experiment, already resolved)

This comparison was run once, locally, against a temporary Chroma store, to select the production
chunk size and overlap. It does not touch the Azure index and does not need to be re-run; it is
retained here for documentation and reproducibility. The outcome of this experiment (800 characters,
150 overlap) is what Section 6 uses to build the production Azure index above.

In [36]:
chunk_configs = [
    {'name': 'small', 'chunk_size': 500,  'chunk_overlap': 75},
    {'name': 'day1',  'chunk_size': 800,  'chunk_overlap': 150},
    {'name': 'large', 'chunk_size': 1100, 'chunk_overlap': 180},
]

experiment_stores = {}

for cfg in chunk_configs:
    experiment_splitter = RecursiveCharacterTextSplitter(
        chunk_size=cfg['chunk_size'],
        chunk_overlap=cfg['chunk_overlap'],
        separators=['\n\n', '\n', '. ', ' ', '']
    )
    experiment_chunks = experiment_splitter.split_documents(pages)
    # Per-document counters so IDs don't collide across sources
    exp_counters = {}
    for chunk in experiment_chunks:
        doc_id = chunk.metadata['document_id']
        exp_counters[doc_id] = exp_counters.get(doc_id, 0) + 1
        chunk.metadata['chunk_id'] = f"{doc_id}-{cfg['name']}-CH-{exp_counters[doc_id]:04d}"
    experiment_stores[cfg['name']] = Chroma.from_documents(
        documents=experiment_chunks,
        embedding=embedding_model,
        collection_name=f"scd_experiment_{cfg['name']}",
        collection_metadata={'hnsw:space': 'cosine'}
    )
    print(f"{cfg['name']:6s}: {len(experiment_chunks)} chunks "
          f"(size={cfg['chunk_size']}, overlap={cfg['chunk_overlap']})")


NameError: name 'Chroma' is not defined

In [ ]:
chunk_test_questions = [
    'When should hydroxyurea therapy be started in adults with sickle cell anemia?',
    'What is the recommended screening approach for stroke risk in children?',
    'How is acute chest syndrome managed?',
]

for question in chunk_test_questions:
    print(f'\n\nQUESTION: {question}')
    for config_name, store in experiment_stores.items():
        print(f'\n--- {config_name.upper()} CONFIGURATION ---')
        results = store.similarity_search_with_relevance_scores(question, k=3)
        for rank, (doc, score) in enumerate(results, start=1):
            print(
                f"{rank}. score={score:.4f} | page={doc.metadata['page_number']} "
                f"| chunk={doc.metadata['chunk_id']}"
            )
            print(doc.page_content[:200].replace('\n', ' '))

### Chunking checkpoint

Choose the configuration that most consistently places complete, relevant evidence near the top.
Change only one variable at a time in any further experiment — a higher similarity score alone
does not prove clinical usefulness.

The Day1 configuration (800 char / 150 overlap) wins clearly across all three questions:

- Hydroxyurea: Day1's rank-1 chunk is the clean numbered recommendation itself. Small's rank-1 is a page-header fragment ("effects of hydroxyurea in males and females EVIDENCE-BASED...") — noise, not content — pushing the real recommendation to rank 2. Large's rank-1 discusses shared decision-making, which is adjacent but not the core threshold criterion.
- Stroke screening: Day1's rank-1 gives the explicit recommendation (annual TCD screening up to age 10, transfusion until 18) — the single most direct answer of any config tested. Small and Large both surface similar content but rank it slightly lower, with more redundant phrasing.
- Acute chest syndrome — the clearest failure case: Small's rank-1 chunk is about splenectomy, and Large's rank-1 chunk is about splenic sequestration — both wrong-topic hits with deceptively high scores. Day1 is the only config whose rank-1 result is actually the ACS background/definition section.

## 16. Build the evaluation set (15–20 questions)

Five categories, per the lab checklist: **direct**, **paraphrased**, **abbreviation/threshold**,
**diagnosis/management-process**, and **out-of-scope**. `in_scope=False` questions test whether
the system correctly refuses instead of forcing an answer from unrelated context.

Fill in `expected_page` / `expected_section` after you've located the real answer in the PDF —
that's what makes the test repeatable rather than post-hoc.

In [37]:
evaluation_questions = [
    # --- Direct (NHLBI guideline) ---
    {'question': 'When should hydroxyurea therapy be started in adults with sickle cell anemia?',
     'category': 'direct', 'in_scope': True, 'expected_section': 'Hydroxyurea Therapy'},
    {'question': 'How is stroke risk screened in children with sickle cell disease?',
     'category': 'direct', 'in_scope': True, 'expected_section': 'Stroke Prevention / TCD'},
    {'question': 'How is acute chest syndrome diagnosed and managed?',
     'category': 'direct', 'in_scope': True, 'expected_section': 'Acute Chest Syndrome'},
    {'question': 'What vaccinations are recommended for infection prevention in sickle cell disease?',
     'category': 'direct', 'in_scope': True, 'expected_section': 'Infection Prevention'},
    {'question': 'What is the recommended management for priapism in sickle cell disease?',
     'category': 'direct', 'in_scope': True, 'expected_section': 'Priapism'},

    # --- Direct (JCM 2024 paper) ---
    {'question': 'How does splenic disease prevalence differ between children and adults with sickle cell disease?',
     'category': 'direct', 'in_scope': True, 'expected_section': 'Splenic Disease'},
    {'question': 'Which age group had the highest ICU admission rate in the Taif multicenter study?',
     'category': 'direct', 'in_scope': True, 'expected_section': 'Hospital Utilisation'},
    {'question': 'How does chronic kidney disease prevalence vary across age groups in sickle cell disease patients?',
     'category': 'direct', 'in_scope': True, 'expected_section': 'Renal Complications'},

    # --- Paraphrased ---
    {'question': 'At what point should a patient with sickle cell anemia begin hydroxyurea?',
     'category': 'paraphrased', 'in_scope': True, 'expected_section': 'Hydroxyurea Therapy'},
    {'question': 'Which imaging test is used to identify children at high risk of stroke?',
     'category': 'paraphrased', 'in_scope': True, 'expected_section': 'Stroke Prevention / TCD'},
    {'question': 'What can be done to lower the risk of alloimmunization from transfusion?',
     'category': 'paraphrased', 'in_scope': True, 'expected_section': 'Transfusion Therapy'},
    {'question': 'Why do middle-aged patients use opioids more frequently than children in sickle cell disease?',
     'category': 'paraphrased', 'in_scope': True, 'expected_section': 'Treatment Adherence'},

    # --- Abbreviation / threshold ---
    {'question': 'What does TCD stand for and what is it used for in sickle cell disease?',
     'category': 'abbreviation', 'in_scope': True, 'expected_section': 'Stroke Prevention / TCD'},
    {'question': 'What velocity on transcranial Doppler indicates elevated stroke risk?',
     'category': 'threshold', 'in_scope': True, 'expected_section': 'Stroke Prevention / TCD'},
    {'question': 'What hemoglobin S percentage threshold is recommended for chronic transfusion targets?',
     'category': 'threshold', 'in_scope': True, 'expected_section': 'Transfusion Therapy'},

    # --- Process / management ---
    {'question': 'What laboratory monitoring is recommended for a patient on hydroxyurea?',
     'category': 'process', 'in_scope': True, 'expected_section': 'Hydroxyurea Therapy'},
    {'question': 'How is renal complication risk assessed in sickle cell disease patients?',
     'category': 'process', 'in_scope': True, 'expected_section': 'Renal Complications'},
    {'question': 'How often should patients be screened for retinopathy?',
     'category': 'process', 'in_scope': True, 'expected_section': 'Ophthalmologic Screening'},

    # --- Out-of-scope (trust / refusal test) ---
    {'question': 'What is the recommended first-line treatment for type 2 diabetes?',
     'category': 'out_of_scope', 'in_scope': False, 'expected_section': None},
    {'question': 'What chemotherapy regimen is used for breast cancer?',
     'category': 'out_of_scope', 'in_scope': False, 'expected_section': None},
    {'question': 'What is the recommended dosage of ibuprofen for a healthy adult with a headache?',
     'category': 'out_of_scope', 'in_scope': False, 'expected_section': None},
]

print(f'Evaluation questions: {len(evaluation_questions)}')
for i, item in enumerate(evaluation_questions, start=1):
    flag = 'IN-SCOPE ' if item['in_scope'] else 'OUT-OF-SCOPE'
    print(f"{i:2d}. [{flag}] [{item['category']}] {item['question']}")


Evaluation questions: 21
 1. [IN-SCOPE ] [direct] When should hydroxyurea therapy be started in adults with sickle cell anemia?
 2. [IN-SCOPE ] [direct] How is stroke risk screened in children with sickle cell disease?
 3. [IN-SCOPE ] [direct] How is acute chest syndrome diagnosed and managed?
 4. [IN-SCOPE ] [direct] What vaccinations are recommended for infection prevention in sickle cell disease?
 5. [IN-SCOPE ] [direct] What is the recommended management for priapism in sickle cell disease?
 6. [IN-SCOPE ] [direct] How does splenic disease prevalence differ between children and adults with sickle cell disease?
 7. [IN-SCOPE ] [direct] Which age group had the highest ICU admission rate in the Taif multicenter study?
 8. [IN-SCOPE ] [direct] How does chronic kidney disease prevalence vary across age groups in sickle cell disease patients?
 9. [IN-SCOPE ] [paraphrased] At what point should a patient with sickle cell anemia begin hydroxyurea?
10. [IN-SCOPE ] [paraphrased] Which imaging

## 17. Manual relevance labeling

For every question, the cell retrieves Top-5 chunks. Read each chunk and type:

- `y` — contains evidence that helps answer the question.
- `n` — unrelated, too vague, or missing the needed evidence.

This is genuinely manual. The code does not infer relevance from score or page number — that
defeats the point of the exercise. Run this cell in Colab and label honestly; it will pause for
input at each chunk.

In [38]:
import sys

manual_evaluation = []

for item in evaluation_questions:
    question = item['question']
    print('\n' + '=' * 110, flush=True)
    print(f"QUESTION [{item['category']}]:", question, flush=True)

    results = retrieve_with_similarity(question, k=5)

    if not item['in_scope']:
        print('OUT-OF-SCOPE CHECK: inspect whether retrieved chunks fail to genuinely support an answer.', flush=True)

    labels = []
    retrieved_rows = []

    for rank, (doc, score) in enumerate(results, start=1):
        print(f"\nRank {rank} | score={score:.4f} | page={doc.metadata['page_number']} "
              f"| section={guess_section(doc.page_content)} | {doc.metadata['chunk_id']}", flush=True)
        print(doc.page_content[:500].replace('\n', ' '), flush=True)

        # Prompt directly within input to ensure immediate display
        label = input('Relevant? (y/n): ').strip().lower()
        while label not in ['y', 'n']:
            label = input('Please enter "y" or "n": ').strip().lower()

        labels.append(1 if label == 'y' else 0)
        retrieved_rows.append({
            'chunk_id': doc.metadata['chunk_id'],
            'page': doc.metadata['page_number'],
            'score': round(score, 4),
        })

    manual_evaluation.append({
        'question': question,
        'category': item['category'],
        'in_scope': item['in_scope'],
        'labels': labels,
        'retrieved': retrieved_rows,
    })

print('\nLabeling complete for', len(manual_evaluation), 'questions.', flush=True)


QUESTION [direct]: When should hydroxyurea therapy be started in adults with sickle cell anemia?

Rank 1 | score=0.0328 | page=78 | section=Hydroxyurea Therapy | nhlbi-scd-2014-CH-0405
 A lack of increase in MCV and/or HbF is not an indication to discontinue therapy.   For the patient who has a clinical response, long-term hydroxyurea therapy is indicated.   Hydroxyurea therapy should be continued during hospitalizations or illness.  EVIDENCE-BASED MANAGEMENT OF SICKLE CELL DISEASE: EXPERT PANEL REPORT, 2014 78
Relevant? (y/n): y

Rank 2 | score=0.0323 | page=75 | section=Hydroxyurea Therapy | nhlbi-scd-2014-CH-0384
Exhibit 12. Evidence Profile—Evidence of Efficacy/Effectiveness for Children and Adults With  Sickle Cell Anemia (Hydroxyurea Versus Usual Care)  Outcome Quality of the Evidence Treatment Effect  Pain crises High Statistically significant benefit  Acute chest syndrome Moderate Statistically significant benefit  Hemoglobin level, fetal hemoglobin level,  need for blood t

## 18. Calculate Precision@3 and Precision@5

$$\text{Precision@K} = \frac{\text{relevant chunks in the first K results}}{K}$$

We average only in-scope questions. Out-of-scope questions are reported separately as a safety
check: any chunk marked relevant there is a sign the system could be fooled into treating
unrelated content as supporting evidence.

In [40]:
def precision_at_k(labels, k):
    return sum(labels[:k]) / k

metric_rows = []
out_of_scope_hits = []

for row in manual_evaluation:
    if row['in_scope']:
        p3 = precision_at_k(row['labels'], 3)
        p5 = precision_at_k(row['labels'], 5)
        metric_rows.append({'question': row['question'], 'category': row['category'],
                             'Precision@3': p3, 'Precision@5': p5})
        print(f"P@3={p3:.2f} | P@5={p5:.2f} | [{row['category']}] {row['question']}")
    else:
        hits = sum(row['labels'])
        out_of_scope_hits.append(hits)
        print(f"OUT-OF-SCOPE | chunks incorrectly marked relevant: {hits}/5 | {row['question']}")

if metric_rows:
    average_p3 = sum(r['Precision@3'] for r in metric_rows) / len(metric_rows)
    average_p5 = sum(r['Precision@5'] for r in metric_rows) / len(metric_rows)
    print(f"\nAverage Precision@3: {average_p3:.3f}")
    print(f"Average Precision@5: {average_p5:.3f}")

if out_of_scope_hits:
    print(f"Average false-relevance rate on out-of-scope questions: "
          f"{sum(out_of_scope_hits) / (len(out_of_scope_hits) * 5):.3f}")

P@3=1.00 | P@5=1.00 | [direct] When should hydroxyurea therapy be started in adults with sickle cell anemia?
P@3=1.00 | P@5=1.00 | [direct] How is stroke risk screened in children with sickle cell disease?
P@3=1.00 | P@5=1.00 | [direct] How is acute chest syndrome diagnosed and managed?
P@3=1.00 | P@5=1.00 | [direct] What vaccinations are recommended for infection prevention in sickle cell disease?
P@3=1.00 | P@5=1.00 | [direct] What is the recommended management for priapism in sickle cell disease?
P@3=1.00 | P@5=1.00 | [direct] How does splenic disease prevalence differ between children and adults with sickle cell disease?
P@3=1.00 | P@5=1.00 | [direct] Which age group had the highest ICU admission rate in the Taif multicenter study?
P@3=0.33 | P@5=0.40 | [direct] How does chronic kidney disease prevalence vary across age groups in sickle cell disease patients?
P@3=0.33 | P@5=0.60 | [paraphrased] At what point should a patient with sickle cell anemia begin hydroxyurea?
P@3=1.00 | P@5

## 19. Retrieval failure log

Name the failure before trying to fix it. Reference table from the workshop:

| Failure mode | Symptom | Fix |
|---|---|---|
| Wrong topic | Medically related, but answers a different question | Better query formulation, metadata filtering, hybrid search |
| Missing context | Chunk has part of a recommendation, not the surrounding criteria | Increase chunk size, add overlap, section-aware chunking |
| Duplicate chunks | Top-K returns near-identical evidence repeatedly | Reduce overlap, deduplicate, diversity retrieval, page/section filtering |
| Exact term missed | Semantic search under-ranks an acronym, drug, or threshold | Keyword search, hybrid retrieval |
| Correct chunk ranked too low | Right chunk exists, but sits at rank 7–8 | Reranking, better chunking, stronger embedding model |
| Irrelevant high-score chunk | High similarity, low clinical relevance | Don't trust score alone; add reranking or human review |
| Metadata problems | Correct text, but page/section missing | Fix the chunking/metadata pipeline — becomes a Day 3 citation problem |

Document at least one **real** case you actually saw above (not a hypothetical).

In [41]:
failure_log = [
    {
        "question": "What is the recommended dosage of ibuprofen for a healthy adult with a headache?",
        "failure_mode": "Wrong topic",
        "symptom": "Retrieved SCD vaso-occlusive crisis (VOC) analgesic protocols containing NSAIDs instead of general adult headache dosing guidelines.",
        "chunk_ids_involved": ["nhlbi-scd-2014-CH-0204", "nhlbi-scd-2014-CH-0208"],
        "proposed_fix": "Better query formulation, metadata filtering, hybrid search",
    },
    {
        "question": "What chemotherapy regimen is used for breast cancer?",
        "failure_mode": "Wrong topic",
        "symptom": "Retrieved SCD transfusion regimens and general preventive screening guidelines (mammography/HPV) due to lexical overlap on 'regimen' and 'breast cancer'.",
        "chunk_ids_involved": ["nhlbi-scd-2014-CH-0674", "nhlbi-scd-2014-CH-0170"],
        "proposed_fix": "Better query formulation, metadata filtering, hybrid search",
    },
    {
        "question": "How often should patients be screened for retinopathy?",
        "failure_mode": "Irrelevant high-score chunk",
        "symptom": "Rank 4 scored high (0.7509) by matching generic pediatric preventive screenings (amblyopia, obesity, HCV) rather than SCD retinal screening intervals.",
        "chunk_ids_involved": ["nhlbi-scd-2014-CH-0165"],
        "proposed_fix": "Don't trust score alone; add reranking or section metadata filtering",
    },
]

for entry in failure_log:
    print(f"[{entry['failure_mode']}] {entry['question']}")
    print(f"  Symptom: {entry['symptom']}")
    print(f"  Chunks: {entry['chunk_ids_involved']}")
    print(f"  Fix: {entry['proposed_fix']}\n")

[Wrong topic] What is the recommended dosage of ibuprofen for a healthy adult with a headache?
  Symptom: Retrieved SCD vaso-occlusive crisis (VOC) analgesic protocols containing NSAIDs instead of general adult headache dosing guidelines.
  Chunks: ['nhlbi-scd-2014-CH-0204', 'nhlbi-scd-2014-CH-0208']
  Fix: Better query formulation, metadata filtering, hybrid search

[Wrong topic] What chemotherapy regimen is used for breast cancer?
  Symptom: Retrieved SCD transfusion regimens and general preventive screening guidelines (mammography/HPV) due to lexical overlap on 'regimen' and 'breast cancer'.
  Chunks: ['nhlbi-scd-2014-CH-0674', 'nhlbi-scd-2014-CH-0170']
  Fix: Better query formulation, metadata filtering, hybrid search

[Irrelevant high-score chunk] How often should patients be screened for retinopathy?
  Symptom: Rank 4 scored high (0.7509) by matching generic pediatric preventive screenings (amblyopia, obesity, HCV) rather than SCD retinal screening intervals.
  Chunks: ['nhlbi-sc

## 20. Keyword and hybrid search

Superseded by the production retrieval path. `retrieve_with_similarity` (Section 6) already
issues a combined keyword and vector query against Azure AI Search on every call, merged
server-side via Reciprocal Rank Fusion. The manually implemented BM25 index that previously
lived in this section is no longer needed and has been removed; hybrid search is now the
default behavior rather than an opt-in experiment.

In [42]:
print('Hybrid search is active by default in retrieve_with_similarity (Section 6). No separate BM25 index required.')

Hybrid search is active by default in retrieve_with_similarity (Section 6). No separate BM25 index required.


In [ ]:
# This cell intentionally left as a no-op.
# The rank_bm25-based manual hybrid experiment previously here has been superseded by
# Azure AI Search's native hybrid retrieval, used by default in retrieve_with_similarity.

## 21. Final retrieval configuration

Based on the Top-K comparison, chunk-config comparison, and Precision@K results above, record
your chosen configuration and the reasoning. This should reference actual numbers from Sections
14, 15, and 18 — not a guess.

In [39]:
FINAL_CONFIG = {
    'backend':       'azure_ai_search',
    'top_k':         7,
    'chunk_size':    800,
    'chunk_overlap': 150,
    'strategy':      'hybrid',  # native Azure AI Search hybrid: BM25 + vector, merged via RRF
}

FINAL_JUSTIFICATION = """
Retrieval configuration, carried forward from the local Chroma-based evaluation and now running
against Azure AI Search:

1. Top-K (k=7): k=5 was generally sufficient, but Section 14 showed relevant evidence for some
   questions, such as vaccination timing, appearing at rank 6 or 7. k=7 captures that evidence
   without reaching the noise-dominated tail observed at k=10.

2. Chunking (800 characters, 150 overlap): this configuration won clearly in the Section 15
   comparison, consistently placing complete, directly relevant evidence at the top rank for
   hydroxyurea, stroke screening, and acute chest syndrome questions. Smaller chunks fragmented
   recommendations; larger chunks sometimes diluted topic precision.

3. Strategy (hybrid): previously recorded as a target strategy without a working implementation,
   since the manually built BM25 index was never wired into the production retrieval function.
   Moving to Azure AI Search resolves this directly: retrieve_with_similarity issues a combined
   keyword and vector query on every call, merged server-side via Reciprocal Rank Fusion, so
   hybrid retrieval is now the actual default behavior rather than an aspirational setting.

Known open item: Azure's Reciprocal Rank Fusion score is on a different numeric scale than the
cosine similarity scores produced by the earlier local Chroma index. RETRIEVAL_THRESHOLD (Day 3,
Section 23) was calibrated against Chroma cosine scores and does not carry over to RRF scores
without fresh calibration against live Azure queries. This is addressed explicitly in Section 23
below rather than left as a silent assumption.
"""

print(FINAL_CONFIG)
print(FINAL_JUSTIFICATION)


{'backend': 'azure_ai_search', 'top_k': 7, 'chunk_size': 800, 'chunk_overlap': 150, 'strategy': 'hybrid'}

Retrieval configuration, carried forward from the local Chroma-based evaluation and now running
against Azure AI Search:

1. Top-K (k=7): k=5 was generally sufficient, but Section 14 showed relevant evidence for some
   questions, such as vaccination timing, appearing at rank 6 or 7. k=7 captures that evidence
   without reaching the noise-dominated tail observed at k=10.

2. Chunking (800 characters, 150 overlap): this configuration won clearly in the Section 15
   comparison, consistently placing complete, directly relevant evidence at the top rank for
   hydroxyurea, stroke screening, and acute chest syndrome questions. Smaller chunks fragmented
   recommendations; larger chunks sometimes diluted topic precision.

3. Strategy (hybrid): previously recorded as a target strategy without a working implementation,
   since the manually built BM25 index was never wired into the produ

### End-of-Day-2 checklist

- [+] Correct evidence frequently appears in Top-K
- [~] Best evidence is reasonably high in the ranking
- [+] Retrieved metadata (document, page, section, chunk ID) is available
- [+] Similarity scores are visible and logged
- [+] A labeled 15–20 question evaluation set exists
- [+] Precision@K is calculated for at least K=3 and K=5
- [+] At least two chunk configurations were compared
- [+] Main failure cases are documented, not just noticed

**Day 3 preview:** Day 2 answered *did we retrieve trustworthy evidence?* Day 3 asks *can the LLM
generate an answer using only that evidence, with a citation back to it?*

# Day 3: Grounded Generation and Citation

Day 2 established a retrieval layer with measured performance (Precision@3 = 0.500, Precision@5 = 0.463, false-relevance rate on out-of-scope questions = 0.000) and a documented failure log. Day 3 adds a constrained generation layer on top of that retrieval: every claim in a generated answer traces back to a specific retrieved chunk, and the system returns a structured refusal when evidence is insufficient or when a question asks for individualized medical judgment rather than guideline information.

Note on the sample starter notebook: the reference "Plan B" notebook (module-based project structure, OpenAI client, simulation mode without an API key) describes a different implementation and is used here as a conceptual reference rather than as source to copy. The cells below build on the existing pipeline already in place: `search_client`, `retrieve_with_similarity`, `guess_section`, `client`, and `LLM_MODEL`.

Section numbering continues from Day 2 (Sections 1-21).

## 22. Day 2 handoff verification

A short check that the retrieval layer from Day 2 is intact in the current session before generation cells depend on it.

In [ ]:
assert 'search_client' in globals(), "Azure AI Search index not connected in this session"
assert 'retrieve_with_similarity' in globals(), "Retrieval function not present in this session"

handoff_question = 'When should hydroxyurea therapy be started in adults with sickle cell anemia?'
handoff_results = retrieve_with_similarity(handoff_question, k=3)
assert len(handoff_results) > 0, "Retrieval returned no results"
print('Day 2 handoff verified against Azure AI Search:', len(handoff_results), 'chunks returned for the check question.')


## 23. Top-K and initial retrieval threshold

Top-K carries over unchanged from the Day 2 final configuration. The retrieval threshold requires
fresh calibration here, because Azure's RRF hybrid score is on a different numeric scale than the
cosine similarity scores the original threshold was set against. The calibration step below runs
one known in-scope question and one known out-of-scope question live, so the threshold is set
against the actual score range this Azure index produces, rather than reused from the earlier
Chroma-based value.

In [60]:
TOP_K_DAY3 = FINAL_CONFIG['top_k']

calibration_in_scope = retrieve_with_similarity(
    'When should hydroxyurea therapy be started in adults with sickle cell anemia?', k=1
)
calibration_out_of_scope = retrieve_with_similarity(
    'What screening interval does this guideline recommend for breast cancer?', k=1
)

in_scope_score = calibration_in_scope[0][1] if calibration_in_scope else None
out_of_scope_score = calibration_out_of_scope[0][1] if calibration_out_of_scope else None

print(f'Top RRF score, known in-scope question:     {in_scope_score}')
print(f'Top RRF score, known out-of-scope question: {out_of_scope_score}')
print('Set RETRIEVAL_THRESHOLD below to a value between these two scores, closer to the out-of-scope score.')

RETRIEVAL_THRESHOLD = 0.025  # set from the calibration output above before running Section 28 onward


Top RRF score, known in-scope question:     0.03279569745063782
Top RRF score, known out-of-scope question: 0.03333333507180214
Set RETRIEVAL_THRESHOLD below to a value between these two scores, closer to the out-of-scope score.


## 24. Citation-ready evidence

Evidence for a question is formatted with every field a citation requires for independent
verification: document, section, page, chunk ID, and the Azure RRF relevance score. The earlier
routing logic that chose between a semantic path and a separate hybrid path has been removed:
`retrieve_with_similarity` is hybrid on every call by default (Section 6), so there is only one
retrieval path to call here.

In [45]:
def prepare_evidence(question: str, k: int = TOP_K_DAY3):
    results = retrieve_with_similarity(question, k=k)
    evidence_blocks = []
    for doc, score in results:
        meta = doc.metadata
        evidence_blocks.append({
            'chunk_id':  meta['chunk_id'],
            'document':  meta['document_id'],
            'citation':  meta['citation'],
            'section':   guess_section(doc.page_content),
            'page':      meta['page_number'],
            'score':     round(score, 4),
            'text':      doc.page_content,
        })
    return evidence_blocks


sample_evidence = prepare_evidence(handoff_question if 'handoff_question' in globals() else handoff_check_question)
for e in sample_evidence[:2]:
    print(f"[{e['chunk_id']}] {e['document']}, {e['section']}, p.{e['page']} (score {e['score']})")
    print(e['text'][:150].replace(chr(10), ' '), '\n')


[nhlbi-scd-2014-CH-0405] nhlbi-scd-2014, Hydroxyurea Therapy, p.78 (score 0.0328)
 A lack of increase in MCV and/or HbF is not an indication to discontinue therapy.   For the patient who has a clinical response, long-term hydroxyu 

[nhlbi-scd-2014-CH-0384] nhlbi-scd-2014, Hydroxyurea Therapy, p.75 (score 0.0323)
Exhibit 12. Evidence Profile—Evidence of Efficacy/Effectiveness for Children and Adults With  Sickle Cell Anemia (Hydroxyurea Versus Usual Care)  Outc 



## 25. Grounding rules

The grounding prompt covers four required parts: a role scoped to citation-bound clinical evidence rather than general medical advice, an explicit boundary against outside knowledge, a required structured output format, and an escape hatch for insufficient evidence. A sixth rule below adds the patient-specific safety boundary needed for Section 27.

In [46]:
GROUNDING_SYSTEM_PROMPT = '''You are a citation-bound clinical evidence assistant for sickle cell disease.

Rules:
1. Answer ONLY using the evidence passages provided below. Do not use outside medical knowledge.
2. Every claim in "recommendation" must be directly supported by the text in "evidence".
3. Return the answer as a JSON object matching exactly this structure:
   {
     "recommendation": "...",
     "evidence": "...",
     "citations": [{"document": "...", "section": "...", "page": N, "chunk_id": "..."}],
     "confidence": "high" | "medium" | "low" | "insufficient"
   }
4. If the evidence does not contain enough information to answer with confidence, confidence is set to "insufficient", "evidence" and "citations" are left empty, and "recommendation" contains a plain refusal rather than a guess.
5. Citations are never invented. A refusal is never softened into a partial guess.
6. Patient-specific medical advice is strictly prohibited. Do not diagnose, prescribe, calculate, recommend, or select a treatment, dose, frequency, duration, monitoring schedule, or treatment change for a specific or implied individual patient.
7. Do not apply general clinical recommendations, thresholds, formulas, weight-based doses, age-based recommendations, laboratory values, or other guideline criteria to the characteristics of an individual patient. General clinical information may be provided only when it is clearly presented as general guidance rather than an individualized recommendation.
8. Treat a question as patient-specific when it provides characteristics of an individual, such as age, weight, symptoms, laboratory values, diagnosis, medical history, current medications, or treatment response, and asks what should be done for that individual. If so, refuse the personalized request even when the supplied evidence contains enough information to answer it.
9. Do not perform arithmetic, calculations, or other transformations on guideline information when doing so would produce a personalized dose, treatment recommendation, monitoring decision, or other clinical decision. When refusing a patient-specific request, set "confidence" to "insufficient" and leave "evidence" and "citations" empty. The "recommendation" should briefly explain that the system provides general clinical information and cannot make individualized medical decisions.
'''
print(GROUNDING_SYSTEM_PROMPT)

You are a citation-bound clinical evidence assistant for sickle cell disease.

Rules:
1. Answer ONLY using the evidence passages provided below. Do not use outside medical knowledge.
2. Every claim in "recommendation" must be directly supported by the text in "evidence".
3. Return the answer as a JSON object matching exactly this structure:
   {
     "recommendation": "...",
     "evidence": "...",
     "citations": [{"document": "...", "section": "...", "page": N, "chunk_id": "..."}],
     "confidence": "high" | "medium" | "low" | "insufficient"
   }
4. If the evidence does not contain enough information to answer with confidence, confidence is set to "insufficient", "evidence" and "citations" are left empty, and "recommendation" contains a plain refusal rather than a guess.
5. Citations are never invented. A refusal is never softened into a partial guess.
6. Patient-specific medical advice is strictly prohibited. Do not diagnose, prescribe, calculate, recommend, or select a treatment

Rule 5 addresses the most common failure mode in ungrounded RAG systems: an answer that states a page number with confidence, where the page, on inspection, does not actually support the claim made. Every citation produced by this pipeline is intended to be one that can be checked by hand, which is exactly what Section 29 automates.

## 26. Structured answer schema

The response schema below is defined inline, since this notebook does not use the sample project's separate `schema/` folder. It enforces the JSON structure from Section 25 and, through the conditional block, enforces rule 4 structurally: a confidence level other than "insufficient" requires non-empty evidence and at least one citation.

In [47]:
import json
from jsonschema import validate, ValidationError

RESPONSE_SCHEMA = {
    "type": "object",
    "required": ["recommendation", "evidence", "citations", "confidence"],
    "properties": {
        "recommendation": {"type": "string"},
        "evidence": {"type": "string"},
        "citations": {
            "type": "array",
            "items": {
                "type": "object",
                "required": ["document", "section", "page", "chunk_id"],
                "properties": {
                    "document": {"type": "string"},
                    "section": {"type": "string"},
                    "page": {"type": "integer"},
                    "chunk_id": {"type": "string"},
                }
            }
        },
        "confidence": {"enum": ["high", "medium", "low", "insufficient"]},
    },
    "allOf": [
        {
            "if": {"properties": {"confidence": {"enum": ["high", "medium", "low"]}}},
            "then": {
                "properties": {
                    "evidence": {"minLength": 1},
                    "citations": {"minItems": 1},
                }
            },
        },
    ],
}

schema_test_good = {
    "recommendation": "Treat with hydroxyurea if three or more moderate to severe pain crises occur in a 12-month period.",
    "evidence": "In adults with SCA who have three or more sickle cell-associated moderate to severe pain crises in a 12-month period, treat with hydroxyurea.",
    "citations": [{"document": "nhlbi-scd-2014", "section": "Hydroxyurea Therapy", "page": 95, "chunk_id": "nhlbi-scd-2014-CH-0421"}],
    "confidence": "high",
}
schema_test_broken = {
    "recommendation": "Take 10mg of drug X daily.",
    "evidence": "",
    "citations": [],
    "confidence": "high",
}

for label, ans in [('Well-formed answer', schema_test_good), ('High confidence with no evidence', schema_test_broken)]:
    try:
        validate(instance=ans, schema=RESPONSE_SCHEMA)
        print(f'{label}: passed validation')
    except ValidationError as e:
        print(f'{label}: rejected ({e.message})')

Well-formed answer: passed validation
High confidence with no evidence: rejected ('' should be non-empty)


The second case is expected to be rejected. A rejection there confirms the schema is doing its job: a "high confidence" answer with zero supporting evidence is exactly the hallucination pattern the grounding rules and schema are designed to catch structurally, independent of the prompt text.

## 27. Patient-specific safety detection

A separate safety layer from the insufficient-evidence path in Section 25, Rule 4. This layer applies even when the source guideline does contain relevant evidence, because population-level guideline evidence is not a substitute for individualized clinical judgment. Requests phrased around a specific individual ("my son", "should I take", "what dose should I give") are redirected to a clinician regardless of retrieval quality.

In [52]:
import re

PATIENT_SPECIFIC_PATTERNS = [
    r'\bmy (son|daughter|child|wife|husband|patient|father|mother)\b',
    r'\bi (have|am|take|was diagnosed)\b',
    r'\bshould i (take|give|start|stop)\b',
    r'\bwhat dose should i\b',
    r'\bcan i give\b',
    r'\bis it safe for me\b',
]

def is_patient_specific(question: str) -> bool:
    lowered = question.lower()
    return any(re.search(p, lowered) for p in PATIENT_SPECIFIC_PATTERNS)

PATIENT_SAFETY_REFUSAL = {
    "recommendation": (
        "I can't provide individualized dosing or treatment instructions "
        "for a specific patient. The appropriate approach should be determined "
        "by a qualified clinician using the patient's clinical assessment."
    ),
    "evidence": "",
    "citations": [],
    "confidence": "insufficient",
}

patient_specific_test_cases = [
    'What dose of hydroxyurea should I give my son?',
    'When should hydroxyurea therapy be started in adults?',
]
for q in patient_specific_test_cases:
    print(q, '->', is_patient_specific(q))

What dose of hydroxyurea should I give my son? -> True
When should hydroxyurea therapy be started in adults? -> False


## 28. Grounded generation

The generation function applies two safety gates before any model call: a patient-specific check (Section 27) and an evidence-sufficiency check against `RETRIEVAL_THRESHOLD` (Section 23). Generation only proceeds once both gates pass, and the model is constrained to JSON output via Groq's structured output mode.

In [53]:
def build_grounded_prompt(question, evidence_blocks):
    context = '\n\n---\n\n'.join(
        f"[chunk_id: {e['chunk_id']} | document: {e['document']} | section: {e['section']} | page: {e['page']}]\n{e['text']}"
        for e in evidence_blocks
    )
    return f'''Evidence:
{context}

Question: {question}

The response is the JSON object described in the system rules, with no additional text.'''


def generate_grounded_answer(question: str, k: int = TOP_K_DAY3):
    if is_patient_specific(question):
        return PATIENT_SAFETY_REFUSAL, None, []

    evidence = prepare_evidence(question, k=k)
    top_score = evidence[0]['score'] if evidence else -1

    if top_score < RETRIEVAL_THRESHOLD:
        insufficient_answer = {
            "recommendation": (
                "The indexed guideline does not appear to cover this topic with enough "
                "confidence for an answer. Rephrasing may help, or the topic may fall "
                "outside this source's scope."
            ),
            "evidence": "",
            "citations": [],
            "confidence": "insufficient",
        }
        return insufficient_answer, None, evidence

    user_prompt = build_grounded_prompt(question, evidence)
    response = client.chat.completions.create(
        model=LLM_MODEL,
        temperature=0,
        max_tokens=1200,
        response_format={"type": "json_object"},
        messages=[
            {'role': 'system', 'content': GROUNDING_SYSTEM_PROMPT},
            {'role': 'user', 'content': user_prompt},
        ],
    )
    raw = response.choices[0].message.content
    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError:
        parsed = {
            "recommendation": "The model response was not valid JSON. Logged as a generation failure in Section 32.",
            "evidence": "", "citations": [], "confidence": "insufficient",
        }
    return parsed, user_prompt, evidence

## 29. Citation verification

Each citation returned by the model is checked against the evidence actually retrieved for the question, so a citation cannot reference a chunk ID outside the retrieved context. A lexical overlap check on the remaining citations flags weak matches for manual review; passing this check is a lightweight signal, not a substitute for reading the source text directly.

In [54]:
def verify_citations(answer, evidence_blocks):
    evidence_by_id = {e['chunk_id']: e for e in evidence_blocks}
    report = []
    for c in answer.get('citations', []):
        cid = c.get('chunk_id')
        if cid not in evidence_by_id:
            report.append((cid, 'fabricated citation: chunk_id not present in retrieved evidence'))
            continue
        real_text = evidence_by_id[cid]['text']
        overlap_ok = any(word in real_text.lower() for word in answer['recommendation'].lower().split()[:6])
        status = 'passed' if overlap_ok else 'flagged for manual review: low lexical overlap'
        report.append((cid, status))
    return report

def show_citation_check(answer, evidence_blocks):
    print('claim:', answer['recommendation'])
    print('confidence:', answer['confidence'])
    print('-' * 100)
    for cid, status in verify_citations(answer, evidence_blocks):
        print(f'  {cid}: {status}')

## 30. Supported, unsupported, and unsafe test cases

Three question categories are run through the same pipeline: a directly supported clinical question, a genuinely out-of-scope question (different disease area, outside the source document), and a patient-specific question that requests personalized dosing.

In [61]:
test_cases = [
    ('supported', 'What does TCD stand for and what is it used for in sickle cell disease?'),
    ('unsupported', 'What is the recommended dosage of ibuprofen for a healthy adult with a headache?'),
    ('unsafe, patient-specific', 'A 14-year-old patient with sickle cell disease weighs 48 kg and has frequent pain crises. Based on the guidelines, what hydroxyurea dose would be appropriate to start, and when should it be increased?'),
]

for label, q in test_cases:
    print(f"\n{'=' * 100}\n[{label}] {q}\n{'=' * 100}")
    answer, prompt_used, evidence = generate_grounded_answer(q)
    print(json.dumps(answer, indent=2))
    try:
        validate(instance=answer, schema=RESPONSE_SCHEMA)
        print('schema: passed')
    except ValidationError as e:
        print('schema: rejected -', e.message)
    if answer['citations']:
        show_citation_check(answer, evidence)


[supported] What does TCD stand for and what is it used for in sickle cell disease?
{
  "recommendation": "TCD stands for Transcranial Doppler, an ultrasound technique used to measure blood flow velocities in the brain. In sickle cell disease it is employed to screen for elevated cerebral blood flow velocities that identify children at high risk for stroke.",
  "evidence": "Transcranial Doppler: An ultrasound technique used to measure blood flow velocities in the brain\u2019s blood. TCD reading is the time averaged mean maximal cerebral blood flow velocity. Children identified as high risk through TCD screening...",
  "citations": [
    {
      "document": "9789240122666",
      "section": "General / Unclassified",
      "page": 1,
      "chunk_id": "9789240122666-CH-0003"
    },
    {
      "document": "nhlbi-scd-2014",
      "section": "Stroke Prevention / TCD",
      "page": 84,
      "chunk_id": "nhlbi-scd-2014-CH-0432"
    },
    {
      "document": "9789240122666",
      "sectio

## 31. Demonstration: answer, claim, citation, retrieved evidence

The required demonstration for the day: one generated claim traced from the answer text through its citation to the literal retrieved chunk that supports it.

In [64]:
demo_answer, demo_prompt, demo_evidence = generate_grounded_answer("What can be done to lower the risk of alloimmunization from transfusion?")

print('answer:')
print(demo_answer['recommendation'])
print('\nclaim -> citation -> retrieved text')
for c in demo_answer['citations']:
    match = next((e for e in demo_evidence if e['chunk_id'] == c['chunk_id']), None)
    print(f"\ncitation: {c}")
    print('retrieved text this citation references:')
    print(match['text'] if match else 'not found in retrieved evidence: fabricated citation')

answer:
To reduce the risk of alloimmunization, transfusions should be used cautiously, especially during the pro‑inflammatory state of acute chest syndrome, and simple transfusion should be avoided in patients whose haemoglobin is 9 g/dL or higher to prevent hyperviscosity that can promote alloimmunisation. In addition, employing more extensive red‑cell antigen matching (e.g., minor RBC matching) for regular transfusions can help lower alloimmunisation risk.

claim -> citation -> retrieved text

citation: {'document': '9789240122666', 'section': 'Transfusion Therapy', 'page': 60, 'chunk_id': '9789240122666-CH-0302'}
retrieved text this citation references:
of recurrent blood transfusion are at increased risk of developing red cell alloantibodies and subsequently 
developing an immunological transfusion reaction. Given that ACS is a significantly inflammatory 
condition, transfusions during this pro-inflammatory state may induce higher rates of alloimmunization 
and hyperviscosity and 

## 32. Generation failure log

A record of one generation-layer failure observed while running Section 30, following the same format as the Day 2 retrieval failure log. Fields are filled in with the actual case observed, not a hypothetical one.

In [65]:
GENERATION_FAILURE_LOG = {
    "question": "When should hydroxyurea therapy be started in adults with sickle cell anemia?",
    "what_happened": "Groq rejected the response with a 400 error (json_validate_failed): max_tokens was reached before the model finished writing valid JSON.",
    "root_cause": "max_tokens=700 was too low for a question whose evidence field required quoting a long retrieved passage, so the JSON object was truncated mid-generation.",
    "fix_applied": "Raised max_tokens from to 1200 in the generate_grounded_answer function.",
    "verified_fixed": True,
}
print(json.dumps(GENERATION_FAILURE_LOG, indent=2))

{
  "question": "When should hydroxyurea therapy be started in adults with sickle cell anemia?",
  "what_happened": "Groq rejected the response with a 400 error (json_validate_failed): max_tokens was reached before the model finished writing valid JSON.",
  "root_cause": "max_tokens=700 was too low for a question whose evidence field required quoting a long retrieved passage, so the JSON object was truncated mid-generation.",
  "fix_applied": "Raised max_tokens from to 1200 in the generate_grounded_answer function.",
  "verified_fixed": true
}


### End-of-Day-3 review

Nine questions from the workshop checklist, answered against the actual output produced in Sections 30-32 rather than in the abstract.

1. Does the answer use only retrieved evidence?
2. Are unsupported claims removed?
3. Does every important claim have a citation?
4. Do the citations support the exact claims made? (Section 29 output)
5. Does the system refuse weak evidence? (unsupported test case, Section 30)
6. Does it refuse patient-specific requests? (unsafe test case, Section 30)
7. Is confidence based on evidence quality rather than the presence of any evidence?
8. Can the answer be traced to the exact source text? (Section 31)
9. Is one failure documented along with its fix? (Section 32)

Known carryover items into Day 4: the retrieval threshold in Section 23 is provisional and needs calibration against full Precision@K data; the hybrid retrieval strategy recorded in the Day 2 final configuration is not yet the default retrieval path used above.